# GIAI ĐOẠN 3: HUẤN LUYỆN `ISOLATION FOREST`, GIẢI THÍCH `SHAP` (XAI) & 4 LỚP KIỂM CHỨNG

**Dự án:** `FinAudit AI — FinForensics` (SE-Hackathon Challenge 2026 — Trường Kinh tế, ĐH Công nghiệp Hà Nội)  
**Mục tiêu Notebook:** Giúp người mới tự theo dõi từng bước xây dựng không gian rủi ro một phía (`One-Sided Risk Space`), huấn luyện `Isolation Forest`, đọc biểu đồ giải thích `SHAP` (đơn vị điểm rủi ro `0–100`) và kiểm chứng khoa học 4 lớp trên toàn thị trường phi tài chính Việt Nam (`2019–2025`).

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, display

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'stage_3_ai_modeling_validation' else Path.cwd()
PROC_DIR = ROOT_DIR / 'data_store' / 'processed'
STAGE_DIR = ROOT_DIR / 'stage_3_ai_modeling_validation'

df = pd.read_parquet(PROC_DIR / 'forensic_scored_full.parquet')
val = json.loads((PROC_DIR / 'stage3_validation_summary.json').read_text(encoding='utf-8'))

print(f'Tổng số dòng đã chấm điểm AI (2019–2025): {val["n_scored_firm_years"]:,} firm-year')
print(f'Độ trung thực giải thích SHAP (R²): {val["layer1_shap_fidelity"]["surrogate_r2"]:.4f} | MAE: {val["layer1_shap_fidelity"]["surrogate_mae_points"]:.2f} điểm')
print(f'Đối chứng thực chứng AUC-ROC: Composite = {val["layer4_empirical_backtest"]["auc_composite"]:.4f} vs Beneish đơn lẻ = {val["layer4_empirical_backtest"]["auc_beneish_only"]:.4f}')

## Bước 1: Kiểm tra Phân bổ Hạng Rủi ro & Bảng Xếp hạng 12 Chỉ số theo Mức Đóng góp `|SHAP|`

In [ ]:
print('=== Phân bổ Hạng Rủi ro Năm 2025 ===')
display(pd.Series(val['risk_tier_counts_2025'], name='Số doanh nghiệp năm 2025').to_frame())

print('=== Tầm quan trọng Toàn cục 12 Đặc trưng theo SHAP ===')
display(pd.DataFrame(val['global_shap_importance']))

## Bước 2: Đối chiếu 9 Doanh nghiệp Tiêu biểu Năm 2025 & Giải trình Tự động từ `SHAP`

In [ ]:
df25 = df[df['year'] == 2025]
spot = ['FPT', 'VNM', 'MWG', 'HPG', 'NVL', 'HAG', 'HVN', 'TTF', 'DIG']
cols = ['ticker', 'icb_sector_vi', 'composite_risk_score', 'risk_tier', 'score_beneish', 'score_altman', 'score_cash_quality', 'score_isoforest', 'top_risk_drivers_vi']
display(df25[df25['ticker'].isin(spot)].set_index('ticker').reindex(spot)[cols[1:]])

## Bước 3: Kiểm tra Chi tiết 4 Lớp Kiểm chứng Khoa học (Ổn định Seed, Out-of-Time & Backtest 24 Doanh nghiệp)

In [ ]:
print('--- Lớp 3A: Độ ổn định khi thay đổi random_state ---')
display(pd.DataFrame(val['layer3_stability']['seed_stability']))

print('--- Lớp 4: Đối chứng 12 Bluechip Minh bạch vs 12 DN Biến cố / Tái cấu trúc ---')
display(pd.DataFrame(val['layer4_empirical_backtest']['group_a_table']).head(12))
display(pd.DataFrame(val['layer4_empirical_backtest']['group_b_table']).head(12))

## Bước 4: Xem Biểu đồ Kiểm chứng & Giải thích `SHAP` 4 Khung

In [ ]:
display(Image(filename=str(STAGE_DIR / 'stage3_validation_dashboard.png')))